<a href="https://colab.research.google.com/github/khoirulanamid/Upscale_Real_ESRGAN/blob/main/4k_Video_Upscaler_Colab_(Real_ESRGAN).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<table>
  <tr>
    <td><img src="https://github.com/xinntao/Real-ESRGAN/raw/master/assets/realesrgan_logo.png" width="120"></td>
    <td><h1>4K Video Upscaler Colab (Real-ESRGAN)</h1></td>
  </tr>
</table>

[![GitHub](https://img.shields.io/badge/GitHub-Repository-blue?logo=github)](https://github.com/khoirulanamid/Upscale_Real_ESRGAN)
[![Real-ESRGAN](https://img.shields.io/badge/AI-Real--ESRGAN-orange)](https://github.com/xinntao/Real-ESRGAN)

---

### 🚀 **Selamat datang di Alat Upscale Media AI**
Notebook ini dirancang untuk mempermudah Anda meningkatkan resolusi **Video & Foto** secara otomatis hingga kualitas **4K** menggunakan algoritma **Real-ESRGAN** yang canggih.

**Author:** [khoirulanamid](https://github.com/khoirulanamid)  
**Credits:** Adapted from the original [Real-ESRGAN](https://github.com/xinntao/Real-ESRGAN) project.

---

# 1. Setup (~1 minute)

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU not detected.. Please change runtime to GPU"

from PIL import Image
import cv2, os, subprocess
from tqdm import tqdm

!git clone https://github.com/xinntao/Real-ESRGAN.git
%cd Real-ESRGAN

!pip install -q torch==2.0.1 torchvision==0.15.2 --extra-index-url https://download.pytorch.org/whl/cu118
!pip install -q basicsr facexlib gfpgan ffmpeg ffmpeg-python
!pip install -q -r requirements.txt
!python setup.py develop

!pip install "numpy<2"
mount_drive = False

# 2. Hubungkan ke Google Drive (Opsional)
Hubungkan Colab ke Drive agar Anda bisa mengambil file video dari Drive atau menyimpan hasil upscale langsung ke sana secara permanen.

*Jika tidak diaktifkan, hasil akan tersimpan di folder sementara dan akan terhapus jika sesi Colab berakhir.*

In [ ]:
# @title Klik untuk Menghubungkan Drive
from google.colab import drive
mount_drive = True #@param{type:"boolean"}

if mount_drive:
    print("⌛ Menghubungkan ke Google Drive...")
    drive.mount('/content/gdrive/')
    print("✅ Terhubung! Anda sekarang bisa menggunakan path '/content/gdrive/MyDrive/'")
else:
    print("⚠️ Drive tidak dihubungkan. Output akan disimpan di penyimpanan sementara.")
```

# 3. Jalankan Upscale (Video atau Foto)
Pilih mode yang ingin Anda gunakan, masukkan path file, lalu jalankan sel ini.

In [ ]:
# @title 3. Panel Kendali Upscale
mode = "Video" #@param ["Video", "Foto"]

# @markdown ---
# @markdown ### 📁 LOKASI OUTPUT
# @markdown *Default: /content/ (Penyimpanan sementara)*\
# @markdown *Gunakan `/content/gdrive/MyDrive/` jika ingin menyimpan ke Google Drive*
output_dir = "/content/" #@param{type:"string"}

# @markdown ---
# @markdown ### 📹 KONFIGURASI VIDEO
video_path = "/content/video.mp4" #@param{type:"string"}
resolution = "FHD (1920 x 1080)" # @param ["FHD (1920 x 1080)", "2k (2560 x 1440)", "4k (3840 x 2160)","2 x original", "3 x original", "4 x original"]
model_video = "RealESRGAN_x4plus" #@param ["RealESRGAN_x4plus" , "RealESRGAN_x4plus_anime_6B", "realesr-animevideov3"]

# @markdown ---
# @markdown ### 🖼️ KONFIGURASI FOTO
image_path = "/content/input.jpg" #@param{type:"string"}
model_foto = "RealESRGAN_x4plus" #@param ["RealESRGAN_x4plus" , "RealESRGAN_x4plus_anime_6B"]

import os, cv2

if not os.path.exists(output_dir):
    os.makedirs(output_dir, exist_ok=True)

if mode == "Video":
    if not os.path.exists(video_path):
        print(f"❌ Error: File video '{video_path}' tidak ditemukan!")
    else:
        video_capture = cv2.VideoCapture(video_path)
        video_width = int(video_capture.get(cv2.CAP_PROP_FRAME_WIDTH))
        video_height = int(video_capture.get(cv2.CAP_PROP_FRAME_HEIGHT))

        final_width, final_height = 1920, 1080
        match resolution:
            case "2k (2560 x 1440)": final_width, final_height = 2560, 1440
            case "4k (3840 x 2160)": final_width, final_height = 3840, 2160
            case "2 x original": final_width, final_height = 2*video_width, 2*video_height
            case "3 x original": final_width, final_height = 3*video_width, 3*video_height
            case "4 x original": final_width, final_height = 4*video_width, 4*video_height

        scale_factor = max(final_width/video_width, final_height/video_height)
        print(f"🚀 Mode Video Aktif")
        print(f"🚀 Memproses: {video_width}x{video_height} -> {final_width}x{final_height}")

        !python inference_realesrgan_video.py -n {model_video} -i '{video_path}' -o '{output_dir}' --outscale {scale_factor}

        video_name = os.path.splitext(os.path.basename(video_path))[0]
        final_video_path = os.path.join(output_dir, f"{video_name}_upscaled.mp4")
        if os.path.exists(f"{output_dir}{video_name}_out.mp4"):
            !mv '{output_dir}{video_name}_out.mp4' '{final_video_path}'
            print(f"✅ Selesai! Video disimpan di: {final_video_path}")

elif mode == "Foto":
    if not os.path.exists(image_path):
        print(f"❌ Error: File foto '{image_path}' tidak ditemukan!")
    else:
        print(f"🚀 Mode Foto Aktif")
        print(f"🚀 Memproses: {image_path}")
        !python inference_realesrgan.py -n {model_foto} -i '{image_path}' -o '{output_dir}' --outscale 4
        print(f"✅ Selesai! Foto hasil upscale ada di folder {output_dir}")
```

# 4. Putuskan Koneksi Otomatis (Opsional)
Aktifkan ini jika Anda ingin sesi Colab otomatis berhenti setelah proses selesai. Ini sangat berguna untuk:
*   **Menghemat Kuota GPU**: Tidak membuang-buang jatah pemakaian saat Anda sedang tidak di depan komputer.
*   **Otomatisasi**: Anda bisa menjalankan proses lalu ditinggal tidur; sistem akan menutup sendiri.

*⚠️ Pastikan Anda sudah menyimpan hasil ke Google Drive (Mount Drive aktif) sebelum menggunakan fitur ini agar file tidak hilang.*

In [ ]:
from google.colab import runtime

disconnect_when_finish = True  #@param{type:"boolean"}

if disconnect_when_finish:
  runtime.unassign()